In [2]:
# !pip install wikipedia
# !pip install transformers==4.46.0
# !pip install torch
# !pip install datasets

In [1]:
import torch
import transformers

In [2]:
import mne
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import time
import re
import torch
import numpy as np
from transformers import GPT2Tokenizer, GPT2Model
from datasets import load_dataset


In [3]:
#load kiloword
path = mne.datasets.kiloword.data_path(
    path=None,        
    force_update=False,  
    update_path=True,     
    download=True       
)

kiloword_data_folder= mne.datasets.kiloword.data_path()
print("Dataset downloaded to:", kiloword_data_folder)

# Epoched-Data
epochs_file = kiloword_data_folder / "kword_metadata-epo.fif"

# Load epoched data
epochs = mne.read_epochs(epochs_file, preload=True)
words = epochs.metadata["WORD"].tolist()

X = epochs.get_data()
n, nchans, ntimes = X.shape

Dataset downloaded to: C:\Users\bolge\mne_data\MNE-kiloword-data
Reading C:\Users\bolge\mne_data\MNE-kiloword-data\kword_metadata-epo.fif ...
Isotrak not found
    Found the data of interest:
        t =    -100.00 ...     920.00 ms
        0 CTF compensation matrices available
Adding metadata with 8 columns
960 matching events found
No baseline correction applied
0 projection items activated


In [4]:

# ── Load pretrained GPT-2 (small: 124M params, 12 layers, 768 dims) ──
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
# output_hidden_states=True → return all 13 layers (0 = token+position embeddings, 1–12 = transformer blocks)
model = GPT2Model.from_pretrained('gpt2', output_hidden_states=True)
model.eval() 


GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2SdpaAttention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)

In [5]:

#takes a sentence and the target word and returns embedding for every word and layer. 
def extract_gpt_embeddings(text, target_word, model = model, tokenizer = tokenizer):
    # 3. Tokenize the input text
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024) # tokenize the raw strings into tokens id, returns pt (pytorch tensor, like numpy array but different)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0]) #converts tokens id into readable strings
    
    
    
    # 4. Find the indices of the target token(s)/target words
    target_indices = [] # this will store only the index of the target word!
    clean_target = target_word.lower()
    
    # Scan through the tokens to find the sequence that builds our word (this is done by simple string matching). later needed for embedding extraction
    for i in range(len(tokens)):
        current_word = ""
        for j in range(i, len(tokens)):
            # Accumulate the string chunks
            current_word += tokens[j].replace('Ġ', '').lower()
            
            # If we perfectly match the target word, save the range of indices
            if current_word == clean_target:
                target_indices = list(range(i, j + 1))
                break
            # If our accumulated string overshoots the target length, stop checking this path
            elif len(current_word) > len(clean_target):
                break
                
        if target_indices:
            break
    
    if not target_indices:
        print(f"Word '{target_word}' not found.")
        return None
    else:
        print(f"Word '{target_word}' maps to subtokens: {[tokens[i] for i in target_indices]}")
        print(f"Token indices: {target_indices}")
        
        # 5. Pass the text through the model
        with torch.no_grad():          #something to save memory
            outputs = model(**inputs)  #passes tokenized sentences through all 13 layers
    
        # 6. Extract the hidden states
        hidden_states = outputs.hidden_states 
    
        # 7. Gather AND Pool embeddings across all layers
        all_layer_embeddings = []                                    #empty list that will collect one vector per layer
        for layer_idx, layer_output in enumerate(hidden_states):     #loops through all layers
            # Extract embeddings for ALL subtokens of our word in this layer 
            #layer output has shape (sentence/batch, n_words/sequence_lenght, hidden size/len of the embedding)
            # Shape becomes [number_of_subtokens, 768]
            subtoken_embeddings = layer_output[0, target_indices, :] # first is the batch(meaning the sentence)
            
            # Average (Mean Pool) them to get a single vector representing the whole word
            word_embedding = torch.mean(subtoken_embeddings, dim=0)
            
            all_layer_embeddings.append(word_embedding)
    
        # 8. Stack into a single tensor
        stacked_embeddings = torch.stack(all_layer_embeddings)
        
        print(f"Shape of final word embeddings: {stacked_embeddings.shape}")

    numpy_embeddings = stacked_embeddings.detach().cpu().numpy() #turns tensor into numpy
    
    print(f"Type: {type(numpy_embeddings)}")
    print(f"Shape of NumPy array: {numpy_embeddings.shape}")
    
    return numpy_embeddings #returns final (13, 768 dim) representing target word in context

# quick test
_ = extract_gpt_embeddings("The quick brown fox jumps over the lazy dog.", "fox")

Word 'fox' maps to subtokens: ['Ġfox']
Token indices: [3]
Shape of final word embeddings: torch.Size([13, 768])
Type: <class 'numpy.ndarray'>
Shape of NumPy array: (13, 768)
Word 'fox' maps to subtokens: ['Ġfox']
Token indices: [3]
Shape of final word embeddings: torch.Size([13, 768])
Type: <class 'numpy.ndarray'>
Shape of NumPy array: (13, 768)


In [ ]:
#!pip install google-genai

In [7]:


import wikipedia
from google import genai

#Gemini fall back. If not enough sentences were found, generate gemini sentences.

client = genai.Client(api_key="GEMINI_API_KEY") # fill in passkey
def generate_synthetic_sentences(word, required_amount):
    print(f"  -> Calling Gemini API for '{word}'...")
    
    # ask for 3x the needed amount to have buffer after filtering
    prompt = f"Write {required_amount * 3} highly diverse, grammatically correct English sentences containing the exact word '{word}'. Return ONLY the sentences, separated by newlines, with absolutely no numbering, bullet points, or extra text."
    
    max_retries = 3  # retries 3 times if API call fails
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt,
            )
            raw_text = response.text.strip()  # gets raw response text
            
            # split response into individual sentences, remove markdown bold markers
            generated_sentences = [s.strip().replace('**', '') for s in raw_text.split('\n') if s.strip()]
            filtered = []
            for sentence in generated_sentences:
                words_in_sent = sentence.split()
                try:
                    word_idx = [w.lower() for w in words_in_sent].index(word.lower())
                except ValueError:
                    continue
                # only keep sentences where target word has at least 4 words of left context
                # needed so GPT-2 has enough context to build a meaningful representation
                if word_idx >= 4:
                    filtered.append(sentence)
                if len(filtered) >= required_amount:  # stops if there are enough sentences
                    break
            
            time.sleep(12)  # waits 12 seconds till next request (rate limit protection)
            return filtered
            
        except Exception as e:
            print(f"  -> API Error (attempt {attempt + 1}/{max_retries}): {e}")
            time.sleep(60)  # waits 60 sec if API call fails before retrying
    
    return []


In [8]:
# ── Collect 10 context sentences per word (3-step fallback pipeline) ──
# 1. WikiText-2 (raw, train)  2. live Wikipedia (top-2 pages)  3. Gemini (synthetic)
# Criteria: target word as separate token with ≥4 words of left context (GPT-2 is causal)
# Returns: results {word: [sentences]} and sentence_table [(word, sentence, source)]
# Output was manually corrected afterwards → sentence_table_fixed.xlsx

def generate_context_sentences(word_list, max_sentences):  
    
    results = {word: [] for word in word_list}
    sentence_table = []
    
    print("Loading Wikipedia dataset...")
    dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
    
    print("\nStarting search pipeline...\n" + "-"*40)
    
    for word in word_list:
        if word.isalnum():
            pattern = re.compile(rf'\b{word}\b', re.IGNORECASE)
        else:
            pattern = re.compile(rf'{word}', re.IGNORECASE)
            
        print(f"Searching for: '{word}'")
        
        # ==========================================
        # ATTEMPT 1: Offline Dataset
        # ==========================================
        for item in dataset:
            text = item['text'].strip()
            if not text:
                continue
                
            sentences = text.split('. ')
            for sentence in sentences:
                if not pattern.search(sentence):
                    continue

                words_in_sent = sentence.strip().split()
                try:
                    word_idx = [w.lower() for w in words_in_sent].index(word.lower())
                except ValueError:
                    continue
                if word_idx < 4:
                    continue

                clean_sentence = sentence.strip()
                if not clean_sentence.endswith('.'):
                    clean_sentence += '.'
                if clean_sentence not in results[word]:
                    results[word].append(clean_sentence)
                    sentence_table.append((word, clean_sentence, 'wikitext'))
                    
                if len(results[word]) >= max_sentences:
                    break

            if len(results[word]) >= max_sentences:
                break
    
        # ==========================================
        # ATTEMPT 2: Live Wikipedia Fallback
        # ==========================================
        if len(results[word]) < max_sentences:
            print(f"  -> Only found {len(results[word])} in dataset. Falling back to live Wikipedia...")
            
            try:
                search_results = wikipedia.search(word, results=2)
                for title in search_results:
                    try:
                        page = wikipedia.page(title, auto_suggest=False)
                        page_sentences = page.content.split('. ')
                        
                        for sentence in page_sentences:
                            if not pattern.search(sentence):
                                continue

                            words_in_sent = sentence.strip().split()
                            try:
                                word_idx = [w.lower() for w in words_in_sent].index(word.lower())
                            except ValueError:
                                continue
                            if word_idx < 4:
                                continue

                            clean_sentence = sentence.strip()
                            if not clean_sentence.endswith('.'):
                                clean_sentence += '.'
                            if clean_sentence not in results[word]:
                                results[word].append(clean_sentence)
                                sentence_table.append((word, clean_sentence, 'wikipedia'))
                                
                            if len(results[word]) >= max_sentences:
                                break

                    except wikipedia.exceptions.DisambiguationError:
                        continue
                    if len(results[word]) >= max_sentences:
                        break

            except Exception as e:
                print(f"  -> Live search failed for '{word}': {e}")
    
        # ==========================================
        # ATTEMPT 3: Gemini Fallback
        # ==========================================
        sentences_needed = max_sentences - len(results[word])
        if sentences_needed > 0:
            print(f"⚠️ '{word}': Missing {sentences_needed} sentences. Generating synthetically...")
            new_sentences = generate_synthetic_sentences(word, sentences_needed)
            results[word].extend(new_sentences)
            for sentence in new_sentences:
                sentence_table.append((word, sentence, 'gemini'))
        
    return results, sentence_table

In [ ]:
# ── Run sentence collection for all 960 words + save raw output) ──

# Rerunning gives different sentences (live Wikipedia + LLM output are not reproducible).

# Raw output: sentence_table.csv, then manually corrected: sentence_table_fixed.xlsx

# all_sentences, sentence_table = generate_context_sentences(words, max_sentences=10)
# sentences_df = pd.DataFrame(sentence_table, columns=['word', 'sentence', 'source'])
# sentences_df.to_csv('sentence_table.csv', index=False)

# import pickle
# with open('all_sentences.pkl', 'wb') as f:
#     pickle.dump(all_sentences, f)

In [9]:
#load sentences and source
sentences_df = pd.read_excel("sentence_table_fixed.xlsx")
sentences_df

,word,sentence,source
0,film,She studied acting under the mentorship of the...,wikitext
1,film,"Tanwar described as a "" boring , exhausting an...",wikitext
2,film,"In 2007 , she was cast in the animated comedy ...",wikitext
3,film,Fey next appeared in the 2010 comedy film Date...,wikitext
4,film,"In 2008 , she starred in the comedy film Baby ...",wikitext
...,...,...,...
9595,solenoid,This is a derivation of the magnetic flux dens...,wikipedia
9596,solenoid,"These solenoids, focus coils, surround nearly ...",wikipedia
9597,solenoid,"""Continuous"" means that the solenoid is not fo...",wikipedia
9598,solenoid,Two of the wires relayed these pulses to the i...,wikitext


In [12]:
#!pip install groq

# ── Fill remaining gaps after manual correction: Llama 3.3 70B via Groq ──
# For words with < 10 sentences after manual cleaning of sentence_table.csv.
# Same prompt and ≥4-word left-context filter as the Gemini fallback.
# Returns updated dict + DataFrame of new rows (source='groq') to append to the sentence table.


from groq import Groq

groq_client = Groq(api_key="GROQ_API")

def fill_missing_sentences_groq(all_sentences_clean, words, max_sentences=10):
    new_rows = []  # collects new sentences for the dataframe
    
    for word in words:
        current = all_sentences_clean.get(word, [])  # get existing sentences for this word
        needed = max_sentences - len(current)  # how many sentences are still missing
        if needed <= 0:
            continue  # word already has enough sentences, skip
        
        print(f"'{word}': {len(current)} found, {needed} missing")
        
        # ask for 3x the needed amount so we have enough after filtering
        prompt = f"Write {needed * 3} highly diverse, grammatically correct English sentences containing the exact word '{word}'. Return ONLY the sentences, separated by newlines, with absolutely no numbering, bullet points, or extra text."
        
        try:
            # call groq api
            response = groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}]
            )
            
            raw_text = response.choices[0].message.content.strip()  # extract text from response
            generated = [s.strip().replace('**', '') for s in raw_text.split('\n') if s.strip()]  # split into individual sentences
            
            filtered = []
            for sentence in generated:
                words_in_sent = sentence.split()
                try:
                    word_idx = [w.lower() for w in words_in_sent].index(word.lower())  # find position of target word
                except ValueError:
                    continue  # target word not found in sentence, skip
                if word_idx >= 4:  # target word must have at least 4 words of left context
                    filtered.append(sentence)
                if len(filtered) >= needed:
                    break  # collected enough sentences
            
            all_sentences_clean[word] = current + filtered  # add new sentences to existing ones
            print(f"  -> Generated {len(filtered)} sentences for '{word}'")  # show how many were added
            
            for sent in filtered:
                new_rows.append({'word': word, 'sentence': sent, 'source': 'groq'})  # log source as groq
        
        except Exception as e:
            print(f"  -> Error: {e}")
    
    return all_sentences_clean, pd.DataFrame(new_rows, columns=['word', 'sentence', 'source'])

In [ ]:
# ── Fill gaps with Groq + save final sentence table (already done — do not rerun) ──
# all_sentences_clean = sentences_df.groupby('word')['sentence'].apply(list).to_dict()
# all_sentences_clean, new_df = fill_missing_sentences_groq(all_sentences_clean, words)
# sentences_df = pd.concat([sentences_df, new_df], ignore_index=True)
#
# # sort rows back into kiloword word order, then overwrite the fixed table
# sentences_df['word'] = pd.Categorical(sentences_df['word'], categories=words, ordered=True)
# sentences_df = sentences_df.sort_values('word').reset_index(drop=True)
# sentences_df.to_excel("sentence_table_fixed.xlsx", index=False)

In [14]:
#prints words that have less than 10 sentences
for word in words:
    n = sentences_df[sentences_df['word'] == word].shape[0]
    if n < 10:
        print(f"'{word}': {n} sentences")

In [15]:
#converts dataframe to dict, groups rows by words.
all_sentences_clean = sentences_df.groupby('word')['sentence'].apply(list).to_dict()

In [ ]:
# ── Extract GPT-2 embeddings (already done — results in gpt2_embeddings.pkl) ──
# For each word: embed it in each of its 10 context sentences (13 layers × 768),
# then average across sentences → one type-level vector per word and layer.
# (cf. Bommasani et al., 2020: averaging contextual embeddings over contexts)
# gpt2_embeddings = {}
#
# for word, sentences in all_sentences_clean.items():
#     word_layer_embeddings = []  # one (13, 768) array per sentence
#     for sentence in sentences:
#         emb = extract_gpt_embeddings(sentence, word)
#         if emb is not None:  # skip sentences where the word could not be located
#             word_layer_embeddings.append(emb)
#     if word_layer_embeddings:
#         gpt2_embeddings[word] = np.mean(word_layer_embeddings, axis=0)  # average over sentences
#
# import pickle
# with open('gpt2_embeddings.pkl', 'wb') as f:
#     pickle.dump(gpt2_embeddings, f)

In [16]:
#load vectors
import pickle
with open('gpt2_embeddings.pkl', 'rb') as f:
    gpt2_embeddings = pickle.load(f)